# Limpieza — Oferta gastronómica (punto 1)

**Cliente:** EcoFood
**Rol:** Punto 1 — Oferta gastronómica y mercado actual
**Fuente:** `data/raw/oferta_gastronomica.csv` — "Oferta y Establecimientos Gastronómicos", Gobierno de la Ciudad de Buenos Aires.

Este notebook toma el CSV crudo y produce `data/processed/establecimientos_clean.csv`.

## Decisiones de limpieza

| # | Problema | Decisión |
|---|---|---|
| 1 | Encoding mixto (latin-1 + UTF-8) | Reparar el mojibake carácter por carácter |
| 2 | Nombres de barrio inconsistentes | Mapear a los 48 barrios oficiales |
| 3 | Columnas con más de 70% de nulos | Eliminar `mail`, `horario`, `calle_cruce` |
| 4 | Nulos en columnas útiles | Etiquetar `SIN DATO`, no borrar filas |
| 5 | 12 duplicados por nombre + dirección | Conservar el registro más completo |
| 6 | No existe una variable "saludable" | Regla documentada sobre `cocina` (opción A) |


## 1. Lectura del archivo

El CSV no es estándar. Tiene tres particularidades:

- separador `;` en vez de coma,
- decimales con coma (`-58,36330675`),
- encoding mixto: la mayor parte está en latin-1, pero algunos valores fueron guardados en UTF-8.

Por eso se lee con `sep=';'`, `encoding='latin-1'` y `decimal=','`.


In [1]:
import unicodedata
import pandas as pd

RAW = '../data/raw/oferta_gastronomica.csv'
PROCESSED = '../data/processed/establecimientos_clean.csv'

df = pd.read_csv(
    RAW,
    sep=';',
    encoding='latin-1',
    decimal=','
)

print('filas:', len(df))
print('columnas:', len(df.columns))
df.head(3)


filas: 2823
columnas: 18


,long,lat,id,nombre,categoria,cocina,ambientacion,telefono,mail,horario,calle_nombre,calle_altura,calle_cruce,direccion_completa,barrio,comuna,codigo_postal,codigo_postal_argentino
0,-58.363307,-34.622963,1602,GUANTANAMERA,RESTAURANTE,CUBANA,MODERNO,4314 0770,NaN,NaN,MOREAU DE JUSTO ALICIA,2092,NaN,"MOREAU DE JUSTO, ALICIA AV. 2092",Puerto Madero,1,1107.0,C1107AFP
1,-58.392330,-34.621554,676,1843 VINACOTECA,VINERIA,NaN,NaN,4779 0806,NaN,lunes a domingo de 8 a 00,HUMBERTO 1º,1843,NaN,HUMBERTO 1º 1843,San Cristobal,3,NaN,NaN
2,-58.453542,-34.560311,487,2080 EXPRESS,RESTAURANTE,TRADICIONAL,CLASICO PORTEÑO,4788 4500,NaN,lunes a domingo de 8 a 00,JURAMENTO,2080,NaN,JURAMENTO AV. 2080,Belgrano,13,1428.0,C1428DNH


## 2. Encoding: reparar el mojibake

Leer todo como latin-1 deja valores como `PORTEÃA`, `ESPAÃA` o `NuÃ±ez`.
Eso es texto que originalmente era UTF-8 y quedó interpretado como latin-1 (o como cp1252). Se repara invirtiendo el proceso, probando los dos encodings y repitiendo hasta que el texto deja de cambiar, porque hay valores doblemente codificados.

Solo se tocan los textos que contienen `Ã` o `Â`, que es la firma del problema.

**Dos casos no se pueden recuperar automáticamente** porque perdieron un carácter en el archivo original: `LA CIG?EÑA` y `PANQUEQUER?A`. El carácter perdido no está en ningún lado, así que se reponen con un mapa de fragmentos dañados, explícito y documentado.


In [2]:
def arreglar_texto(valor):
    """Repara texto UTF-8 que quedó leído como latin-1 o como cp1252."""
    if not isinstance(valor, str):
        return valor
    for _ in range(4):
        if 'Ã' not in valor and 'Â' not in valor:
            break
        for encoding in ('latin-1', 'cp1252'):
            try:
                candidato = valor.encode(encoding).decode('utf-8')
            except (UnicodeEncodeError, UnicodeDecodeError):
                continue
            if candidato != valor:
                valor = candidato
                break
        else:
            break
    return valor


COLUMNAS_TEXTO = [
    'nombre', 'categoria', 'cocina', 'ambientacion', 'calle_nombre',
    'calle_cruce', 'direccion_completa', 'barrio', 'codigo_postal_argentino',
    'telefono',
]

antes = df[COLUMNAS_TEXTO].astype(str).apply(lambda c: c.str.contains('Ã|Â').sum())
df[COLUMNAS_TEXTO] = df[COLUMNAS_TEXTO].map(arreglar_texto)

# Caracteres que se perdieron en el archivo de origen (quedaron como � o ?).
# No hay forma de deducirlos: se reponen a mano y queda documentado.
FRAGMENTOS_DANADOS = {
    'Â�': 'Ü',
    'Ã‘': 'Ñ',
    'Ã?': 'Í',
}
for columna in COLUMNAS_TEXTO:
    for danado, correcto in FRAGMENTOS_DANADOS.items():
        df[columna] = df[columna].str.replace(danado, correcto, regex=False)

despues = df[COLUMNAS_TEXTO].astype(str).apply(lambda c: c.str.contains('Ã|Â|�').sum())

pd.DataFrame({'antes': antes, 'despues': despues})


,antes,despues
nombre,69,1
categoria,0,0
cocina,71,0
ambientacion,0,0
calle_nombre,33,0
calle_cruce,0,0
direccion_completa,31,0
barrio,28,0
codigo_postal_argentino,0,0
telefono,0,0


## 3. Barrios: normalizar a los 48 oficiales

El dataset usa nombres cortos y sin tildes (`Boca`, `Paternal`, `Velez Sarsfield`), y uno de los barrios quedó escrito como `NuÃ±ez` por el problema de encoding.

Se comparan los nombres ignorando tildes, puntos, mayúsculas y espacios, y se reemplazan por el nombre oficial.

> **Aviso — Parque Chas:** después de normalizar aparecen los 48 barrios oficiales, pero **Parque Chas no tiene ningún registro**. No es un error de limpieza: el barrio no figura en el dataset. Hay que tenerlo presente antes de leer "baja oferta" como oportunidad.


In [3]:
BARRIOS_OFICIALES = [
    'Agronomía', 'Almagro', 'Balvanera', 'Barracas', 'Belgrano', 'Boedo', 'Caballito',
    'Chacarita', 'Coghlan', 'Colegiales', 'Constitución', 'Flores', 'Floresta', 'La Boca',
    'La Paternal', 'Liniers', 'Mataderos', 'Monte Castro', 'Monserrat', 'Nueva Pompeya',
    'Núñez', 'Palermo', 'Parque Avellaneda', 'Parque Chacabuco', 'Parque Chas',
    'Parque Patricios', 'Puerto Madero', 'Recoleta', 'Retiro', 'Saavedra', 'San Cristóbal',
    'San Nicolás', 'San Telmo', 'Vélez Sársfield', 'Versalles', 'Villa Crespo',
    'Villa del Parque', 'Villa Devoto', 'Villa Gral. Mitre', 'Villa Lugano', 'Villa Luro',
    'Villa Ortúzar', 'Villa Pueyrredón', 'Villa Real', 'Villa Riachuelo',
    'Villa Santa Rita', 'Villa Soldati', 'Villa Urquiza',
]


def clave(texto):
    """Minúsculas, sin tildes, sin puntos ni espacios: sirve para comparar nombres."""
    sin_tildes = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return sin_tildes.lower().replace('.', '').replace(' ', '')


oficial_por_clave = {clave(b): b for b in BARRIOS_OFICIALES}
# Alias: el dataset usa el nombre corto de estos dos barrios.
oficial_por_clave['boca'] = 'La Boca'
oficial_por_clave['paternal'] = 'La Paternal'

df['barrio'] = df['barrio'].map(lambda b: oficial_por_clave.get(clave(b), b))

print('barrios distintos:', df['barrio'].nunique())
print('valores sin mapear:', sorted(set(df['barrio']) - set(BARRIOS_OFICIALES)))

conteo = df['barrio'].value_counts()
sin_registro = [b for b in BARRIOS_OFICIALES if b not in conteo.index]
print('barrios oficiales sin ningún registro:', sin_registro)


barrios distintos: 47
valores sin mapear: []
barrios oficiales sin ningún registro: ['Parque Chas']


## 4. Duplicados

Hay 12 pares con el mismo `nombre` y la misma `direccion_completa`.

No se pueden borrar por `id`, porque son ids distintos. La regla es:

1. Contar cuántos campos completos tiene cada fila.
2. Ordenar por cantidad de campos (de mayor a menor) y, en empate, por `id` (de menor a mayor).
3. Quedarse con la primera fila de cada par nombre + dirección.

En 4 pares el `cocina` difiere entre los dos registros (`TAO-TAO` figura como JAPONESA y como CHINA). La regla elige uno solo, y eso se documenta como limitación.


In [4]:
df['_campos_completos'] = df.notna().sum(axis=1)
filas_antes = len(df)

df = (
    df.sort_values(
        ['nombre', 'direccion_completa', '_campos_completos', 'id'],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(subset=['nombre', 'direccion_completa'], keep='first')
    .drop(columns='_campos_completos')
    .reset_index(drop=True)
)

print(f'filas: {filas_antes} -> {len(df)} (eliminadas {filas_antes - len(df)})')


filas: 2823 -> 2811 (eliminadas 12)


## 5. Nulos

La regla es no perder información: **un local existe aunque no esté clasificado**.

- Se eliminan solo las columnas que no sirven para el análisis por su nivel de nulos: `mail` (83%), `horario` (74%) y `calle_cruce` (99,8%).
- En el resto de las columnas de texto, el nulo se reemplaza por `SIN DATO`.
- `codigo_postal` se pasa a texto para que no quede como `1107.0`.


In [5]:
COLUMNAS_INUTILES = ['mail', 'horario', 'calle_cruce']
df = df.drop(columns=COLUMNAS_INUTILES)

COLUMNAS_TEXTO = [
    'nombre', 'categoria', 'cocina', 'ambientacion', 'telefono',
    'calle_nombre', 'direccion_completa', 'codigo_postal_argentino',
]
df[COLUMNAS_TEXTO] = df[COLUMNAS_TEXTO].fillna('SIN DATO')

df['codigo_postal'] = (
    df['codigo_postal'].astype('Int64').astype('string').fillna('SIN DATO')
)

nulos = df.isnull().sum()
print('columnas con nulos restantes:')
print(nulos[nulos > 0] if (nulos > 0).any() else 'ninguna')


columnas con nulos restantes:
ninguna


## 6. Tipos de datos

- `id`, `calle_altura`: enteros.
- `comuna`: entero nullable (`Int8`), para no gastar memoria y dejar explícito que es una categoría numérica.
- `long` y `lat`: float (ya quedaron bien por `decimal=','`).


In [6]:
df['id'] = df['id'].astype('int64')
df['comuna'] = df['comuna'].astype('Int8')
df['calle_altura'] = df['calle_altura'].astype('int64')
df['long'] = df['long'].astype('float64')
df['lat'] = df['lat'].astype('float64')

df.dtypes


long                       float64
lat                        float64
id                           int64
nombre                         str
categoria                      str
cocina                         str
ambientacion                   str
telefono                       str
calle_nombre                   str
calle_altura                 int64
direccion_completa             str
barrio                         str
comuna                        Int8
codigo_postal               string
codigo_postal_argentino        str
dtype: object

## 7. Variables derivadas

### 7.1 Oferta saludable (opción A)

El dataset no trae una variable "saludable". Se construye a partir de `cocina`, con una regla explícita y conservadora:

- **Núcleo estricto:** `NATURISTA`, `VEGETARIANA`.
- **Criterio amplio:** el núcleo + `MEDITERRANEA`, `SUSHI`, `JAPONESA`.

Se usa comparación exacta contra la lista, no búsqueda de substrings: si se busca `FIT` aparece `CONFITERIA`, y si se busca `ECO` aparece `EL RECOLETO`.

> **Limitación:** el núcleo estricto son muy pocos registros (0,4% del total). El criterio amplio se usa para el análisis geográfico, pero el núcleo estricto se reporta aparte para no exagerar la conclusión.

### 7.2 Zona

Se agrupan las 15 comunas en 4 zonas para poder contrastar los hallazgos a nivel agregado:

| Zona | Comunas |
|---|---|
| Centro | 1, 3 |
| Norte | 2, 12, 13, 14 |
| Oeste | 6, 7, 11, 15 |
| Sur | 4, 5, 8, 9, 10 |

> **Ojo con la comuna 12:** agrupa Saavedra, Villa Urquiza, Villa Pueyrredón y Coghlan. En el primer borrador del mapa no estaba y dejaba 70 registros sin zona. Va a **Norte**. Se detectó al construir el EDA, cuando esos barrios aparecían con zona vacía.


In [7]:
SALUDABLE_ESTRICTO = {'NATURISTA', 'VEGETARIANA'}
SALUDABLE_AMPLIO = SALUDABLE_ESTRICTO | {'MEDITERRANEA', 'SUSHI', 'JAPONESA'}

cocina_norm = df['cocina'].str.upper().str.strip()

df['es_saludable_estricto'] = cocina_norm.isin(SALUDABLE_ESTRICTO)
df['es_saludable_amplio'] = cocina_norm.isin(SALUDABLE_AMPLIO)

print('saludables estrictos:', int(df['es_saludable_estricto'].sum()))
print('saludables amplios:  ', int(df['es_saludable_amplio'].sum()))
print()
print(df.loc[df['es_saludable_amplio'], 'cocina'].value_counts())


saludables estrictos: 11
saludables amplios:   80

cocina
MEDITERRANEA    48
SUSHI           12
JAPONESA         9
NATURISTA        7
VEGETARIANA      4
Name: count, dtype: int64


In [8]:
ZONA_POR_COMUNA = {
    1: 'Centro', 3: 'Centro',
    2: 'Norte', 12: 'Norte', 13: 'Norte', 14: 'Norte',
    6: 'Oeste', 7: 'Oeste', 11: 'Oeste', 15: 'Oeste',
    4: 'Sur', 5: 'Sur', 8: 'Sur', 9: 'Sur', 10: 'Sur',
}

df['zona'] = df['comuna'].map(ZONA_POR_COMUNA).astype('category')

print('registros sin zona:', int(df['zona'].isna().sum()))
df['zona'].value_counts()


registros sin zona: 0


zona
Centro    1090
Norte      999
Sur        363
Oeste      359
Name: count, dtype: int64

## 8. Validación de calidad

Antes de guardar se chequea que el dataset cumpla las condiciones mínimas.

Queda **un solo valor** con un carácter dañado: el nombre `OTTO G?RGUELL`. El carácter original no está en el archivo (quedó reemplazado por `�`), así que no se puede reconstruir sin adivinar. No se usa en el análisis y se deja tal cual, documentado.


In [9]:
checks = {
    'filas': len(df),
    'barrios distintos': df['barrio'].nunique(),
    'barrio nulo': int(df['barrio'].isnull().sum()),
    'comuna nula': int(df['comuna'].isnull().sum()),
    'registros sin zona': int(df['zona'].isna().sum()),
    'comunas fuera de 1-15': int((~df['comuna'].between(1, 15)).sum()),
    'lat fuera de CABA': int((~df['lat'].between(-34.71, -34.52)).sum()),
    'long fuera de CABA': int((~df['long'].between(-58.55, -58.34)).sum()),
    'duplicados nombre+direccion': int(df.duplicated(subset=['nombre', 'direccion_completa']).sum()),
    'mojibake restante': int(
        df[COLUMNAS_TEXTO].astype(str)
          .apply(lambda c: c.str.contains('Ã|Â|�').sum()).sum()
    ),
}
pd.Series(checks)


filas                          2811
barrios distintos                47
barrio nulo                       0
comuna nula                       0
registros sin zona                0
comunas fuera de 1-15             0
lat fuera de CABA                 0
long fuera de CABA                0
duplicados nombre+direccion       0
mojibake restante                 1
dtype: int64

In [10]:
mascara = df[COLUMNAS_TEXTO].astype(str).apply(lambda c: c.str.contains('Ã|Â|�'))
for columna in COLUMNAS_TEXTO:
    valores = df.loc[mascara[columna], columna].unique()
    if len(valores):
        print(columna, list(valores))


nombre ['OTTO Gâ€�RGUELL']


## 9. Guardar el dataset limpio

Se guarda en UTF-8, sin índice, para que cualquier integrante lo lea sin volver a pelear con el encoding.

También se deja un archivo aparte con los barrios oficiales que no tienen ningún registro (hoy: Parque Chas), para que el aviso no se pierda cuando el dataset pase a la etapa de EDA.


In [11]:
df.to_csv(PROCESSED, index=False, encoding='utf-8')

comuna_por_barrio = df.drop_duplicates('barrio').set_index('barrio')['comuna'].to_dict()
comuna_por_barrio.setdefault('Parque Chas', 15)

sin_registro = pd.DataFrame({
    'barrio': [b for b in BARRIOS_OFICIALES if b not in set(df['barrio'])],
})
sin_registro['comuna'] = sin_registro['barrio'].map(comuna_por_barrio)
sin_registro.to_csv('../data/processed/barrios_sin_registro.csv', index=False)

print('guardado:', PROCESSED)
print('barrios sin registro:')
print(sin_registro.to_string(index=False))


guardado: ../data/processed/establecimientos_clean.csv
barrios sin registro:
     barrio  comuna
Parque Chas      15


## 10. Resumen y limitaciones

**Resultado:** 2.823 registros crudos → 2.811 limpios, 15 columnas, 47 barrios con datos.

**Decisiones tomadas:**

- Encoding reparado: 232 valores dañados en 5 columnas (69 nombres, 71 cocinas, 33 calles, 31 direcciones, 28 barrios), más 2 caracteres perdidos repuestos a mano.
- Barrios normalizados a los 48 oficiales; Parque Chas queda sin registros.
- 12 duplicados eliminados por nombre + dirección.
- 3 columnas eliminadas por nulos altos; el resto de los nulos pasó a `SIN DATO`.
- `es_saludable_estricto` y `es_saludable_amplio` construidas con regla explícita.
- `zona` agregada a partir de la comuna.

**Limitaciones a declarar en el informe:**

1. El núcleo estricto de oferta saludable son 11 registros (0,4%): el registro oficial casi no clasifica este segmento.
2. En 4 duplicados el `cocina` era contradictorio y se conservó un solo valor.
3. Parque Chas no aparece: no se puede afirmar que "no tiene oferta", solo que no está registrado.
4. El dataset es una foto con fecha de descarga, no una serie temporal.
5. Un nombre de local conserva un carácter dañado irreconstruible (`OTTO G?RGUELL`); no afecta el análisis.
